# 05 — Vector Databases

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/05_vector_databases.ipynb)

## 📌 What is it?
Vector databases store embeddings and support fast approximate nearest neighbor (ANN) search at scale. ChromaDB, FAISS, Pinecone, Weaviate, and Qdrant are the main players.

## 🧠 Why AI Engineers need this
RAG pipelines require storing thousands to millions of document embeddings and querying for the most relevant ones in milliseconds. This is what vector DBs are built for.

In [ ]:
# ── FAISS — Facebook's fast ANN library (local) ──
# pip install faiss-cpu

faiss_code = '''
import faiss
import numpy as np

# ── BUILD AN INDEX ──
dim = 1536      # embedding dimension (OpenAI text-embedding-3-small)
n_docs = 10000  # number of documents

# Generate mock embeddings
np.random.seed(42)
embeddings = np.random.randn(n_docs, dim).astype(np.float32)
faiss.normalize_L2(embeddings)   # normalize for cosine similarity

# FlatL2 — exact search (use for < 100K vectors)
index = faiss.IndexFlatIP(dim)   # Inner Product ≈ cosine after normalization
index.add(embeddings)
print(f"Index size: {index.ntotal} vectors")

# ── SEARCH ──
query = np.random.randn(1, dim).astype(np.float32)
faiss.normalize_L2(query)

k = 5
distances, indices = index.search(query, k)
print(f"Top-{k} results: indices={indices[0]}, scores={distances[0].round(3)}")

# ── SAVE / LOAD ──
faiss.write_index(index, "my_index.faiss")
loaded_index = faiss.read_index("my_index.faiss")
print(f"Loaded index with {loaded_index.ntotal} vectors")
'''
print("FAISS Code Pattern:")
print(faiss_code)

In [ ]:
# ── CHROMADB — easiest to use, great for development ──
# pip install chromadb

chroma_code = '''
import chromadb
from chromadb.utils import embedding_functions

# In-memory (dev) or persistent (prod)
client = chromadb.Client()                         # in-memory
# client = chromadb.PersistentClient(path="./db")  # persistent

# Create collection
collection = client.create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}   # use cosine similarity
)

# Add documents (ChromaDB handles embedding with the right setup)
collection.add(
    ids=["doc1", "doc2", "doc3"],
    documents=[
        "Python is a high-level programming language",
        "Machine learning enables computers to learn from data",
        "RAG combines retrieval with language generation"
    ],
    metadatas=[
        {"source": "intro.txt", "topic": "python"},
        {"source": "ml.txt",    "topic": "ml"},
        {"source": "rag.txt",   "topic": "ai"}
    ]
)

# Query
results = collection.query(
    query_texts=["What is RAG?"],
    n_results=2,
    where={"topic": "ai"}   # metadata filtering!
)
print(results["documents"])
print(results["distances"])
print(results["metadatas"])
'''
print("ChromaDB Code Pattern:")
print(chroma_code)

In [ ]:
# ── VECTOR DB COMPARISON ──
comparison = {
    "ChromaDB":  {
        "type": "Open-source, local or cloud",
        "scale": "< 1M vectors comfortably",
        "best_for": "Development, prototyping, small RAG apps",
        "setup": "pip install chromadb",
        "cost": "Free"
    },
    "FAISS": {
        "type": "Library (Facebook), local only",
        "scale": "Millions-Billions (GPU version)",
        "best_for": "Research, high-throughput, no infrastructure",
        "setup": "pip install faiss-cpu",
        "cost": "Free"
    },
    "Pinecone": {
        "type": "Managed cloud service",
        "scale": "Billions of vectors",
        "best_for": "Production at scale, managed infra",
        "setup": "pip install pinecone-client",
        "cost": "Paid (free tier available)"
    },
    "Qdrant": {
        "type": "Open-source, self-hosted or cloud",
        "scale": "Millions-Billions",
        "best_for": "Production, rich filtering, open-source",
        "setup": "Docker or pip install qdrant-client",
        "cost": "Free (self-hosted)"
    },
    "Weaviate": {
        "type": "Open-source, self-hosted or cloud",
        "scale": "Billions",
        "best_for": "Multi-modal, GraphQL API",
        "setup": "Docker + pip install weaviate-client",
        "cost": "Free (self-hosted)"
    },
}

print(f"{'Name':<12} {'Type':<35} {'Scale':<30} {'Best For'}")
print("-" * 120)
for name, info in comparison.items():
    print(f"{name:<12} {info['type']:<35} {info['scale']:<30} {info['best_for']}")

print("\n💡 Recommendation: Start with ChromaDB → migrate to Pinecone/Qdrant for production")

## 🔗 What's Next?
[06 — LangChain Basics →](06_langchain_basics.ipynb)